# 15 — LLM-as-a-Judge and Human Evaluation

    ## Scenario and success criteria

    A support team calibrates a rubric judge against human labels before using it for low-risk triage.

    This guided lab succeeds when its assertions pass and the learner can explain why the baseline fails, what the mitigation changes, and which production controls remain outside the simulation.

    ## Learning objectives

    - Write observable rubric criteria.
- Measure judge–human agreement.
- Route ambiguous and high-impact cases to people.

    **Prerequisites:** Courses 01–13 and the preceding advanced/enterprise lesson.
    **Safety boundary:** all behavior is deterministic and synthetic; there are no credentials, external calls, or side effects. Printed results are simulation evidence, not a live-model benchmark.

## Mental model and architecture

![Course 15 architecture](diagram-1.svg)

Treat the model as one uncertain component inside a deterministic control plane. Inputs, schemas, identity, authorization, metrics, release gates, and state transitions remain application responsibilities.

## Baseline and failure injection

A plausible judge score is not ground truth and may contain position, style, or self-preference bias.

The next cell defines the synthetic fixture and the smallest reusable primitive needed to make that failure observable.

In [ ]:
from lab15 import agreement, pairwise_winner, requires_human_review, rubric_judge

responses = {
    "weak": "We received your message.",
    "strong": "We are sorry. We will replace the mug immediately.",
}
judgements = [rubric_judge(case_id, text) for case_id, text in responses.items()]
human_scores = {"weak": 1, "strong": 5}

## Experiment

Run the baseline and candidate on the same fixture so the comparison is attributable.

In [ ]:
print(judgements)
print("exact agreement", agreement(judgements, human_scores))
print("pairwise winner", pairwise_winner(responses["weak"], responses["strong"]))
for judgement in judgements:
    print(judgement.case_id, "human review:", requires_human_review(judgement, high_impact=False))

## Evaluation

The assertions below are the executable contract. They validate both a positive path and a boundary or failure path; a printed claim alone is not proof.

In [ ]:
assert agreement(judgements, human_scores) == (2, 2)
assert pairwise_winner(responses["weak"], responses["strong"]) == "right"
assert requires_human_review(rubric_judge("ambiguous", "Sorry."), high_impact=False)

## Production upgrade

Calibrate on held-out, double-scored examples; log rubric version and reason codes; periodically re-check disagreement and subgroup slices. Never request or store hidden chain-of-thought.

| Teaching lab | Production system |
| --- | --- |
| Synthetic fixtures | Versioned, reviewed, privacy-safe datasets |
| Deterministic simulation | Provider adapter plus optional recorded replay |
| In-process state | Durable state with tenant and retention boundaries |
| Assertions | CI gates, staged rollout, monitoring, and rollback |

## Exercises

1. Add one normal, one boundary, and one adversarial case without weakening an invariant.
2. Change one design variable and report the metric numerator, denominator, unit, and direction.
3. Write a production decision memo that identifies owner, failure policy, monitoring signal, and rollback trigger.

## Takeaway

Use probabilistic components for bounded interpretation; use trusted deterministic code for permissions, validation, metrics, and consequential state changes.